In [1]:
import numpy as np
from pathways import Pathways

for scenario in [
    "remind-SSP2-NPi-stem-SPS1.zip",
    #"remind-SSP2-PkBudg1150-stem-SPS1.zip",
    #"remind-SSP2-PkBudg500-stem-SPS1.zip",
    #"remind-SSP2-NPi-stem-SPS4.zip",
    #"remind-SSP2-PkBudg1150-stem-SPS4.zip",
    #"remind-SSP2-PkBudg500-stem-SPS4.zip",
]:
    p = Pathways(datapackage=scenario, debug=False)
    
    p.calculate(
        methods=[
            'EF v3.1 EN15804 - climate change - global warming potential (GWP100)',
            #'EF v3.1 EN15804 - material resources: metals/minerals - abiotic depletion potential (ADP): elements (ultimate reserves)',
            #'EF v3.1 EN15804 - ecotoxicity: freshwater - comparative toxic unit for ecosystems (CTUe)',
            #'EF v3.1 EN15804 - land use - soil quality index',
            #'EF v3.1 EN15804 - water use - user deprivation potential (deprivation-weighted water consumption)',
            #'EF v3.1 EN15804 - particulate matter formation - impact on human health',

        ] 
        ,
        regions=["CH",],
        scenarios=p.scenarios.pathway.values.tolist(),
        variables=[v for v in p.scenarios.coords["variables"].values if v.startswith("FE")],
        years=[
            #2020,
            #2030,
            #2040,
            2050
        ],
        multiprocessing=True,
        use_distributions=0,
        # subshares=True,
        
    )
    p.export_results()
    del p

Calculating LCA results for remind...
--- Calculating LCA results for SSP2-NPi - SPS1...
------ Calculating LCA results for 2050...
Results exported to results_20250826_125919.gzip


In [ ]:
p = Pathways(datapackage=scenario, debug=False)

In [1]:
[m for m in p.lcia_methods if "abiotic" in m]

NameError: name 'p' is not defined

In [12]:
import pandas as pd
fp="results_20250826_125919.gzip"
df = pd.read_parquet(fp, engine='pyarrow')

In [13]:
df = df[df["value"]!=0.0]
df = df[~df["value"].isnull()]
df=df.reset_index()

In [14]:
from premise.geomap import Geomap
geo = Geomap("image")
EU = geo.iam_to_ecoinvent_location("WEU")
EU.remove("CH")

In [15]:
df.loc[df["location"].isin(EU), "location"] = "EU wo CH"
df.loc[~df["location"].isin(["CH", "EU wo CH"]), "location"] = "RoW"

In [16]:
df = df.groupby([
    'variable', 'year', 'region', 'model', 'scenario', 'impact_category', 'location', "act_category",
]).sum().reset_index()
df = df[['variable', 'year', 'region', 'model', 'scenario', 'impact_category', 'location', "value"]]
print(len(df))

16909


In [17]:
df.head()

,variable,year,region,model,scenario,impact_category,location,value
0,FE_bus_diesel,2050,CH,remind,SSP2-NPi - SPS1,EF v3.1 EN15804 - climate change - global warm...,CH,8.095542e-05
1,FE_bus_diesel,2050,CH,remind,SSP2-NPi - SPS1,EF v3.1 EN15804 - climate change - global warm...,CH,9.190071e-07
2,FE_bus_diesel,2050,CH,remind,SSP2-NPi - SPS1,EF v3.1 EN15804 - climate change - global warm...,CH,1.499336e-05
3,FE_bus_diesel,2050,CH,remind,SSP2-NPi - SPS1,EF v3.1 EN15804 - climate change - global warm...,CH,1.705617e-03
4,FE_bus_diesel,2050,CH,remind,SSP2-NPi - SPS1,EF v3.1 EN15804 - climate change - global warm...,CH,-4.188689e-03


In [25]:
df.loc[
    (df["year"] == 2050)
    &(df["impact_category"].str.contains("climate")),
    "value"
].sum()/1e9

1.5088976692643419

In [18]:
a = df.groupby([
    'variable',
    'year',
    'region',
    'model',
    'scenario',
    'impact_category',
    'location',
]).sum()["value"].to_xarray()

In [9]:
df = a.interp(year=np.arange(2020, 2051)).to_dataframe("value").reset_index()

In [19]:
df = df[df["value"]!=0.0]
df = df[~df["value"].isnull()]
df=df.reset_index()

In [20]:
len(df)

16909

In [21]:
df["impact_category"] = df["impact_category"].str.replace("RELICS - metals extraction - ", "")
df["impact_category"] = df["impact_category"].str.replace("EF v3.1 - climate change - global warming potential (GWP100)", "climate change")
df["impact_category"] = df["impact_category"].str.replace("EF v3.1 EN15804 - climate change - global warming potential (GWP100)", "climate change")
df["impact_category"] = df["impact_category"].str.replace("selected LCI results - resource - land occupation", "land occupation")
df["impact_category"] = df["impact_category"].str.replace("EF v3.1 - material resources: metals/minerals - abiotic depletion potential (ADP): elements (ultimate reserves)", "natural resources (minerals and metals)")
df["impact_category"] = df["impact_category"].str.replace("IPCC 2021 - climate change - GWP 100a, incl. H and bio CO2", "climate change")
df["impact_category"] = df["impact_category"].str.replace("Cumulative Energy Demand (CED) - energy resources: renewable - energy content (HHV)", "renewable primary energy")
df["impact_category"] = df["impact_category"].str.replace("Cumulative Energy Demand (CED) - energy resources: non-renewable - energy content (HHV)", "non-renewable primary energy")
df["impact_category"] = df["impact_category"].str.replace("ReCiPe 2016 v1.03, endpoint (I) - total: ecosystem quality - ecosystem quality", "ecosystem impact")
df["impact_category"] = df["impact_category"].str.replace("ReCiPe 2016 v1.03, endpoint (E) - total: human health - human health", "human health")
df["impact_category"] = df["impact_category"].str.replace("Ecological Scarcity 2021 - total - UBP", "ecological scarcity")

In [22]:
units = {
    "climate change": "kg CO2-eq.",
    "land occupation": "square meter-year",
    "natural resources (minerals and metals)": "kg Sb-eq.",
    "renewable primary energy": "MJ (HHV)",
    "non-renewable primary energy": "MJ (HHV)",
    "ecosystem impact": "species-year lost",
    "human health": "DALY",
    "ecological scarcity": "UBP pt"
}
df["unit"] = df["impact_category"].map(units)
df.loc[df["unit"].isnull(), "unit"] = "kg"

In [64]:
pivot = df.pivot_table(index=['variable', 'region', 'model', 'scenario', 'impact_category', 'location', 'unit'], 
                          columns='year', 
                          values='value', 
                          aggfunc='sum').reset_index()
pivot.fillna(0, inplace=True)

scenarios = p.scenarios.sel(region="CH", pathway=p.scenarios.pathway.values.tolist()[0], model="remind").astype(float).interp(
    year=range(p.scenarios.coords["year"].values.min(), p.scenarios.coords["year"].values.max() + 1)
)

df_scenarios = scenarios.to_dataframe("value").reset_index().pivot_table(
    index=['variables', 'region', 'model', 'pathway', ], 
                          columns='year', 
                          values='value', 
                          aggfunc='sum'
)

df_scenarios["unit"] = "PJ"
df_scenarios["impact_category"] = "Production"

df_scenarios = df_scenarios.reset_index()
cols = ['variables', 'region', 'model', 'pathway', "impact_category"] + list(np.arange(2020, 2050))
df_scenarios=df_scenarios.loc[:, cols]

df_scenarios.rename(columns={'variables':'variable'}, inplace=True)
df_scenarios.rename(columns={'pathway':'scenario'}, inplace=True)

df_final = pd.concat([pivot, df_scenarios], ignore_index=True)
df_final.loc[df_final["location"].isnull(), "location"] = "CH"
df_final.to_excel("indicators_export_SPS1.xlsx")

In [59]:
df_scenarios.loc[:, :2050]

year,variable,region,model,scenario,2005,2006,2007,2008,2009,2010,...,2041,2042,2043,2044,2045,2046,2047,2048,2049,2050
0,Biomass CHP,CH,remind,SSP2-NPi-SPS1,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,Biomass IGCC,CH,remind,SSP2-NPi-SPS1,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,Biomass IGCC CCS,CH,remind,SSP2-NPi-SPS1,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,CAES,CH,remind,SSP2-NPi-SPS1,0.0,0.0,0.0,0.0,0.0,0.0,...,0.016927,0.025139,0.033352,0.041564,0.049777,0.057989,0.066202,0.074414,0.082627,0.090839
4,Carbon dioxide removal_nan_Electricity,CH,remind,SSP2-NPi-SPS1,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
285,wind,CH,remind,SSP2-NPi-SPS1,0.0,0.0,0.0,0.0,0.0,0.0,...,8.513843,9.287861,10.061878,10.835895,11.609913,12.383930,13.157948,13.931965,14.705983,15.480000
286,wood (forest),CH,remind,SSP2-NPi-SPS1,0.0,0.0,0.0,0.0,0.0,0.0,...,15.310558,17.433550,19.556542,21.679534,23.802525,25.925517,28.048509,30.171501,32.294493,34.417485
287,wood (landscape maintenance),CH,remind,SSP2-NPi-SPS1,0.0,0.0,0.0,0.0,0.0,0.0,...,6.168529,5.956433,5.744338,5.532243,5.320148,5.108053,4.895957,4.683862,4.471767,4.259672
288,wood (residues),CH,remind,SSP2-NPi-SPS1,0.0,0.0,0.0,0.0,0.0,0.0,...,9.969367,9.707945,9.446524,9.185103,8.923681,8.662260,8.400838,8.139417,7.877996,7.616574


In [2]:
import numpy as np
np.show_config()

Build Dependencies:
  blas:
    detection method: pkgconfig
    found: true
    include directory: /opt/homebrew/Caskroom/miniforge/base/envs/pathways/include
    lib directory: /opt/homebrew/Caskroom/miniforge/base/envs/pathways/lib
    name: blas
    openblas configuration: unknown
    pc file directory: /opt/homebrew/Caskroom/miniforge/base/envs/pathways/lib/pkgconfig
    version: 3.9.0
  lapack:
    detection method: internal
    found: true
    include directory: unknown
    lib directory: unknown
    name: dep4377784592
    openblas configuration: unknown
    pc file directory: unknown
    version: 1.26.4
Compilers:
  c:
    args: -ftree-vectorize, -fPIC, -fstack-protector-strong, -O2, -pipe, -isystem,
      /opt/homebrew/Caskroom/miniforge/base/envs/pathways/include, -fdebug-prefix-map=/Users/runner/miniforge3/conda-bld/numpy_1707225640867/work=/usr/local/src/conda/numpy-1.26.4,
      -fdebug-prefix-map=/opt/homebrew/Caskroom/miniforge/base/envs/pathways=/usr/local/src/conda-pre

In [9]:
from pivottablejs import pivot_ui
from IPython.display import HTML
pivot_ui(df, outfile_path='pivottable_ch3.html')